In [3]:
import os
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Text Normalization for Bilingual Data (focusing on Arabic text rules)
def clean_bilingual_text(text: str) -> str:
    # Remove Arabic diacritics (Tashkeel)
    tashkeel_pattern = re.compile(r"[\u064B-\u0652]")
    text = re.sub(tashkeel_pattern, "", text)
    
    # Normalize Alef variants to bare Alef
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)
    
    # Normalize Ta Marbuta to Ha
    text = re.sub(r"\u0629", "\u0647", text)
    
    # Clean whitespace and structural artifacts
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 2. Ingestion and Processing Pipeline
class DocumentProcessor:
    def __init__(self):
        # Using a highly capable multilingual model optimized for semantic search
        self.model = SentenceTransformer('sentence-transformers/LaBSE')
        self.index = None
        self.documents = []

    def process_and_index(self, raw_documents: list):
        """
        Expects a list of dicts: [{'id': 1, 'text': '...', 'lang': 'en'}]
        """
        cleaned_texts = []
        for doc in raw_documents:
            cleaned = clean_bilingual_text(doc['text'])
            self.documents.append({'id': doc['id'], 'text': cleaned, 'lang': doc['lang']})
            cleaned_texts.append(cleaned)
            
        # Generate multilingual vector embeddings
        embeddings = self.model.encode(cleaned_texts, convert_to_numpy=True)
        
        # Build FAISS index for Cognitive Search capabilities
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings)
        print(f"Successfully indexed {len(cleaned_texts)} bilingual document chunks.")

    def cognitive_search(self, query: str, top_k: int = 1):
        cleaned_query = clean_bilingual_text(query)
        query_vector = self.model.encode([cleaned_query], convert_to_numpy=True)
        distances, indices = self.index.search(query_vector, top_k)
        
        results = []
        for idx in indices[0]:
            if idx != -1:
                results.append(self.documents[idx])
        return results

# Quick pipeline smoke test execution
if __name__ == "__main__":
    sample_data = [
        {"id": 101, "text": "This Agreement is entered into between Company X and Partner Y.", "lang": "en"},
        {"id": 102, "text": "تم إبرام هذه الاتفاقية بين الشركة س والشريك ص لتنظيم الخدمات البنكية.", "lang": "ar"}
    ]
    processor = DocumentProcessor()
    processor.process_and_index(sample_data)


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--LaBSE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Successfully indexed 2 bilingual document chunks.
